### Rain-on-snow project <br>
#### USGS and NWM streamflow time series retrieval

This notebook retrieves NWM retrospective streamflow for GAGES II (reference and non-reference basins) using TEEHR python package.<br>

Docummentation on TEEHR usage can be found in:<br>
https://rtiinternational.github.io/teehr/

In [ ]:
import geopandas as gpd
import pandas as pd
import teehr
import teehr.fetching.nwm.retrospective_points as nwm_retro
from pathlib import Path
from datetime import datetime
import numpy as np

In [2]:
# NOTE:
# The download component retrieves pre-processed data from the TEEHR data warehouse via the TEEHR-HUB REST API. 
# Starting in May 2026 the API now requires authentication through the use of API keys or bearer tokens, which can be obtained by contacting the TEEHR team.
# API Key was requested to the team set up in the shell. See documentation in https://rtiinternational.github.io/teehr/user_guide/fetching.html#configure-the-api

# Check it it is properly set with the lines below
import os
print(bool(os.getenv("TEEHR_DOWNLOAD_API_KEY")))

True


1. User specific definitions

In [ ]:
######### local paths
# base directory where input metadata is located
base_dir = Path('../input')

# output directory for retrieved data
output_dir = Path('../output')

5. Fetch NWM retrospective streamflow timeseries from AWS

In [ ]:
# OPTIONAL:  start a dask cluster (not required but will run much faster - requires installing dask.distributed)

import dask
from dask.distributed import LocalCluster

# Run this to allow you to see the dask dashboard at the Client dashboard link if on teehr hub:
#dask.config.set({"distributed.dashboard.link": "{JUPYTERHUB_SERVICE_PREFIX}proxy/{port}/status"})
dask.config.set({"distributed.dashboard.link": "http://{host}:{port}/status"})
#dask.config.refresh()

cluster = LocalCluster()
client = cluster.get_client()
client

In [ ]:
data_dir = Path(base_dir, "nwm_retro_archive_uvm")
locations_path = Path(base_dir, 'Metadata_GAGESII_ROS_wComid.parquet')

# Select sites to process
df_locs = pd.read_parquet(locations_path, engine='pyarrow')
df_locs_fltr = df_locs[df_locs['in_nwm']]
nwm_ids = df_locs_fltr['comid'].unique().tolist()

In [ ]:
# Definition of input variables
nwm_version = "CONUS"
variable_name = "streamflow"
start_date="1979-10-01"
end_date="1979-10-02"

In [ ]:
nwm_retro.nwm_retro_to_parquet(
       nwm_version = nwm_version,
       variable_name = variable_name,
       start_date = start_date,
       end_date = end_date,
       location_ids = nwm_ids,
       domain = "CONUS",
       output_parquet_dir = Path(data_dir, "nwm30_retrospective"),
       )

In [ ]:
# Example code to fecth from warehouse
#ev.fetch.nwm_retrospective_points(
#    nwm_version="nwm30",
#    variable_name="streamflow",
#    start_date="1979-10-01",
#    end_date="2022-10-01",
#    #chunk_by="month",          # bounds memory within each water year
#    domain="CONUS",
#    table_name="secondary_timeseries",
#    #write_mode="append"#"upsert"
#)

Full CONUS data is processed using scripts in `HPC_Scripts/teehr_nwm_fetch`. Once the full period is processed, follow with the section below.<br>
If processing the files locally with the function above is not very time consuing, it could be done without using an HPC.

5.1 Create a single NWM retrospective file for the GAGES II ref and non-ref basins

In [1]:
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import pandas as pd
import datetime as dt

In [ ]:
# Combine all the yearly parquet files into a single file by STREAMING
#---------------------------------------------------------------------
# The directory holds 44 yearly TEEHR files.
# Loading them all with dataset.to_table().to_pandas() materializes every row in RAM and
# crashes the kernel. Instead we stream record batches straight from the dataset to a
# ParquetWriter, so memory stays flat regardless of dataset size. Timestamps are written
# exactly as stored in the source files (naive, no timezone attached).
configuration = "nwm30_retrospective"
in_dir = Path(output_dir, "nwm_retro_archive_uvm", configuration)

dataset = ds.dataset(in_dir, format="parquet")   # 44 yearly files = one logical dataset

# Single combined file, written outside the source dir so a re-run's ds.dataset(in_dir)
# never re-globs the combined file back in as an input.
out_path = Path(output_dir, configuration, "nwm_retro_gagesii_conus.parquet")
out_path.parent.mkdir(parents=True, exist_ok=True)

writer = pq.ParquetWriter(out_path, dataset.schema)
n_rows = 0
try:
    for batch in dataset.scanner(batch_size=1_000_000).to_batches():
        writer.write_batch(batch)
        n_rows += batch.num_rows
finally:
    writer.close()

print(f"Combined {n_rows:,} rows -> {out_path}")

Combined 3,414,440,671 rows -> output/nwm30_retrospective/nwm_retro_gagesii_conus.parquet


In [6]:
# Time-range diagnostics (streaming - reads only the two time columns, never the whole frame)
#---------------------------------------------------------------------------------------------
import pyarrow.compute as pc

vmin = vmax = None
ref_uniques = set()
for batch in dataset.scanner(columns=["value_time", "reference_time"],
                             batch_size=2_000_000).to_batches():
    vt = batch.column("value_time")
    bmin, bmax = pc.min(vt).as_py(), pc.max(vt).as_py()
    vmin = bmin if vmin is None else min(vmin, bmin)
    vmax = bmax if vmax is None else max(vmax, bmax)
    ref_uniques.update(batch.column("reference_time").unique().to_pylist())

print("value_time range:", vmin, "->", vmax)
print("reference_time uniques:", sorted(u for u in ref_uniques if u is not None)[:10],
      "(+NaT)" if None in ref_uniques else "")

value_time range: 1979-02-01 01:00:00 -> 2022-12-31 00:00:00
reference_time uniques: [] (+NaT)


In [ ]:
# Let's look at the header to double-check
out_path = f'{output_dir}/nwm30_retrospective/nwm_retro_gagesii_conus.parquet'

# Reads just the first batch of 5 rows, not the whole file
first5 = next(pq.ParquetFile(out_path).iter_batches(batch_size=5)).to_pandas()
first5

,reference_time,value_time,value,variable_name,configuration_name,unit_name,location_id,member,created_at,updated_at
0,NaT,1979-02-01 01:00:00,20.750000,streamflow,nwm30_retrospective,m3 s-1,nwm30-721640,None,NaT,NaT
1,NaT,1979-02-01 01:00:00,22.400000,streamflow,nwm30_retrospective,m3 s-1,nwm30-724696,None,NaT,NaT
2,NaT,1979-02-01 01:00:00,21.189999,streamflow,nwm30_retrospective,m3 s-1,nwm30-805443,None,NaT,NaT
3,NaT,1979-02-01 01:00:00,7.100000,streamflow,nwm30_retrospective,m3 s-1,nwm30-811537,None,NaT,NaT
4,NaT,1979-02-01 01:00:00,40.200001,streamflow,nwm30_retrospective,m3 s-1,nwm30-805113,None,NaT,NaT
